# Импорт библиотек:

In [1]:
import serial
import serial.tools.list_ports
import time
import numpy as np
import plotly
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import pandas as pd
from IPython.display import display, clear_output
from ipywidgets import Button, Output

# Получаем список всех доступных портов:

In [2]:
ports = serial.tools.list_ports.comports()

for port in ports:
    print(port.device, port.manufacturer, port.description)

COM12 FTDI USB Serial Port (COM12)


# Читаем данные с СОМ-порта:

In [3]:
ports = serial.tools.list_ports.comports()

for port in ports:
    print(port.device, port.manufacturer, port.description)

COM12 FTDI USB Serial Port (COM12)


In [20]:
new_list = [[1],[2],[3],[4]]
new_line = [val for sublist in new_list for val in sublist]
new_line.insert(0,0.2)
with open('data0_4.csv', 'w') as file:
    file.write(str(new_line))

pd.read_csv('data0_4.csv').head(10)

,[0.2,1,2,3,4]


### Функция декодирования пакетов

In [21]:
def decode_packet(packet):
    # Декодирование одного пакета
    if len(packet) != 5:
        return "Invalid packet. Packet length does not equals '5'."
    
    sync, msb, mid, lsb, checksum = packet
    
    # Проверка синхробайта
    if sync not in [0xA0, 0xA1, 0xA2, 0xA3]:
        return "Invalid packet. Sync byte error."
    
    # Проверка контрольной суммы
    if checksum != (msb ^ mid ^ lsb):
        return "Invalid packet. Checksum error."
    
    # Декодирование значения
    value = (msb << 16) | (mid << 8) | lsb
    channel = sync - 0xA0 + 1
    
    return channel, value

### Задаём параметры

In [22]:
# Параметры
port = "COM12"
baudrate = 115200
dt = 0.01  # интервал обновления
max_points = 1000  # максимальное количество точек на графике

### Создаём графики

In [23]:
# Создание графиков
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Канал 1', 'Канал 4'),
    vertical_spacing=0.1
)

# Добавление трасс
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=2, col=1)

fig.update_layout(
    height=800,
    showlegend=False,
    title_text="Данные с последовательного порта"
)


# Настройка осей
#for i in range(1, 5):
#    fig.update_xaxes(title_text="Время (с)", range=[0, max_points * dt], row=i, col=1)
#    fig.update_yaxes(title_text="Значение", range=[0, 100_000_000], row=i, col=1)

# Настройка осей для двух графиков
for i in range(1, 3):
    fig.update_xaxes(title_text="Время (с)", range=[0, max_points * dt], row=i, col=1)
    fig.update_yaxes(title_text="Значение", row=i, col=1)

In [25]:
df = pd.DataFrame(columns=['0','1','2','3','4'])

# Создание виджета
fig_widget = go.FigureWidget(fig)
display(fig_widget)

print(f"Connecting to port: {port}...")
try:
    ser = serial.Serial(port, baudrate, timeout=1)
    print("Connected!")
except serial.SerialException as e:
    print(f"Error connecting to {port}: {e}")
    exit()

buffer = bytearray()

# Хранение данных для графиков
data_lists = [[], [], [], []]  # список списков для каждого канала
time_list = []  # общий список времени

t = 0.0  # начальное время

with open('data0_4.csv', 'w') as file:

    try:
        while True:
            # Чтение данных из порта
            if ser.in_waiting:
                raw_data = ser.read(ser.in_waiting)
                buffer.extend(raw_data)
            
            # Обработка буфера
            i = 0
            packets_processed = 0
            new_line = [0,0,0,0,0]
            
            while i <= len(buffer) - 5:
                if buffer[i] in [0xA0, 0xA1, 0xA2, 0xA3]:
                    packet = buffer[i:i+5]
                    result = decode_packet(packet)
                    if result:
                        channel, value = result
                        new_line[channel] = value
                        if channel == 4:
                            new_line[0] = t
                            new_line = ",".join(map(str, new_line))
                            file.write(new_line + '\n')
                            new_line = [0,0,0,0,0]
                            
                        #print(f"Channel {channel}: {value}")

                        # Добавляем значение в соответствующий список
                        channel_idx = channel - 1
                        data_lists[channel_idx].append(value)

                        
                        
                        # Ограничиваем размер списка
                        if len(data_lists[channel_idx]) > max_points:
                            data_lists[channel_idx] = data_lists[channel_idx][-max_points:]
                        
                        # Обновляем время
                        time_list.append(t)
                        if len(time_list) > max_points:
                            time_list = time_list[-max_points:]
                        
                        t += dt
                        packets_processed += 1
                        i += 5
    
                        continue
                i += 1
    
            # Удаляем обработанные данные из буфера
            if i > 0:
                buffer = buffer[i:]
            
            # ОБНОВЛЕНИЕ ГРАФИКОВ - КЛЮЧЕВОЙ МОМЕНТ
            # Создаем новые данные для каждого графика
            #for ch in range(4):
            if data_lists:
                ch_time = [i * 0.01 for i in range(len(data_lists[0]))]
                fig_widget.data[0].x = ch_time
                fig_widget.data[1].x = ch_time
                fig_widget.data[0].y = data_lists[0]
                fig_widget.data[1].y = data_lists[3]


            
            # Принудительное обновление виджета (не всегда нужно, но помогает)
            #fig_widget.update_layout(yaxis=(0, max(max(data_lists[ch]))))
            
            # Небольшая пауза для снижения нагрузки CPU
            time.sleep(0.001)
        
    except KeyboardInterrupt:
        print("\nStopping...")
    except serial.SerialException as se:
        print(f"Serial port error: {se}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        ser.close()
        print("Port closed")

# Вывод статистики после остановки
print(f"\nStatistics:")
for i in range(4):
    print(f"Channel {i+1}: {len(data_lists[i])} points")

FigureWidget({
    'data': [{'mode': 'lines',
              'type': 'scatter',
              'uid': '119f7407-f19b-4a28-a97b-25645f51d824',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'mode': 'lines',
              'type': 'scatter',
              'uid': '878a9120-8009-4920-b939-4a0751bee14c',
              'x': [],
              'xaxis': 'x2',
              'y': [],
              'yaxis': 'y2'}],
    'layout': {'annotations': [{'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Канал 1',
                                'x': 0.5,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                          

Connecting to port: COM12...
Connected!

Stopping...
Port closed

Statistics:
Channel 1: 1000 points
Channel 2: 1000 points
Channel 3: 1000 points
Channel 4: 1000 points


In [ ]:
df2 = pd.read_csv("data.csv", header=None)
df2.head(50)

In [26]:
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [27]:
x = np.arange(0, 100)

def measure(x):
    yield [np.sin(0.2 * x), np.cos(0.4 * x)]

def f(x):
    return np.sin(x)

In [28]:
test_data = []
for i in range(100):
    test_data.append(f(i))
test_data

[np.float64(0.0),
 np.float64(0.8414709848078965),
 np.float64(0.9092974268256817),
 np.float64(0.1411200080598672),
 np.float64(-0.7568024953079282),
 np.float64(-0.9589242746631385),
 np.float64(-0.27941549819892586),
 np.float64(0.6569865987187891),
 np.float64(0.9893582466233818),
 np.float64(0.4121184852417566),
 np.float64(-0.5440211108893698),
 np.float64(-0.9999902065507035),
 np.float64(-0.5365729180004349),
 np.float64(0.4201670368266409),
 np.float64(0.9906073556948704),
 np.float64(0.6502878401571168),
 np.float64(-0.2879033166650653),
 np.float64(-0.9613974918795568),
 np.float64(-0.7509872467716762),
 np.float64(0.14987720966295234),
 np.float64(0.9129452507276277),
 np.float64(0.8366556385360561),
 np.float64(-0.008851309290403876),
 np.float64(-0.8462204041751706),
 np.float64(-0.9055783620066238),
 np.float64(-0.13235175009777303),
 np.float64(0.7625584504796027),
 np.float64(0.956375928404503),
 np.float64(0.27090578830786904),
 np.float64(-0.6636338842129675),
 np.fl

### Вариант от DeepSeek

In [29]:
data = [np.random.randint(0,100) for _ in range(100)]

fig = go.FigureWidget(go.Scatter(
    x=[],
    y=[],
    mode='lines'
))

display(fig)

for i in range(100):
    data.pop(0)
    data.append(np.random.randint(0,100))

    fig.data[0].x = list(range(len(data)))
    fig.data[0].y = data

    time.sleep(0.1)

FigureWidget({
    'data': [{'mode': 'lines', 'type': 'scatter', 'uid': 'fa3b1583-1605-4d1c-90ff-a5b860d680b6', 'x': [], 'y': []}],
    'layout': {'template': '...'}
})

In [30]:
# Инициализация данных для 4 графиков
data1 = [np.random.randint(0, 100) for _ in range(100)]
data2 = [np.random.randint(0, 100) for _ in range(100)]
data3 = [np.random.randint(0, 100) for _ in range(100)]
data4 = [np.random.randint(0, 100) for _ in range(100)]

# Создаем подграфики 2x2
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('График 1', 'График 2', 'График 3', 'График 4'),
    vertical_spacing=0.1
)

# Добавляем трассы для каждого графика
fig.add_trace(go.Scatter(x=list(range(100)), y=data1, mode='lines', name='График 1'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data2, mode='lines', name='График 2'), row=2, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data3, mode='lines', name='График 3'), row=3, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data4, mode='lines', name='График 4'), row=4, col=1)

# Настройка внешнего вида
fig.update_layout(
    title='4 графика в реальном времени',
    height=1200,
    showlegend=False,
    template='plotly_white'
)

# Настройка осей для каждого подграфика
fig.update_xaxes(title_text="Индекс", row=1, col=1)
fig.update_xaxes(title_text="Индекс", row=2, col=1)
fig.update_xaxes(title_text="Индекс", row=3, col=1)
fig.update_xaxes(title_text="Индекс", row=4, col=1)
fig.update_yaxes(title_text="Значение", row=1, col=1)
fig.update_yaxes(title_text="Значение", row=2, col=1)
fig.update_yaxes(title_text="Значение", row=3, col=1)
fig.update_yaxes(title_text="Значение", row=4, col=1)

# Устанавливаем диапазоны для всех осей
for i in range(1, 5):
    fig.update_xaxes(range=[0, 100], row=i, col=1)
    fig.update_yaxes(range=[0, 100], row=i, col=1)

# Преобразуем в FigureWidget для интерактивного обновления
fig = go.FigureWidget(fig)

# Отображаем график
display(fig)

# Обновляем данные в цикле
for i in range(100):
    # Обновляем данные
    data1.pop(0)
    data1.append(np.random.randint(0, 100))
    
    data2.pop(0)
    data2.append(np.random.randint(0, 100))
    
    data3.pop(0)
    data3.append(np.random.randint(0, 100))
    
    data4.pop(0)
    data4.append(np.random.randint(0, 100))
    
    # Обновляем графики
    with fig.batch_update():
        fig.data[0].x = list(range(len(data1)))
        fig.data[0].y = data1
        
        fig.data[1].x = list(range(len(data2)))
        fig.data[1].y = data2
        
        fig.data[2].x = list(range(len(data3)))
        fig.data[2].y = data3
        
        fig.data[3].x = list(range(len(data4)))
        fig.data[3].y = data4
        
        # Обновляем заголовок с номером итерации
        fig.layout.title.text = f'4 графика в реальном времени (обновление {i+1}/100)'
    
    time.sleep(0.1)

print("Обновление завершено!") 

FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'График 1',
              'type': 'scatter',
              'uid': '29ca027d-75c5-47f7-82dd-364362b3adfe',
              'x': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17,
                    18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
                    34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
                    50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
                    66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81,
                    82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97,
                    98, 99],
              'xaxis': 'x',
              'y': [80, 94, 29, 93, 47, 55, 69, 26, 18, 0, 94, 23, 85, 16, 56, 49,
                    3, 95, 59, 9, 36, 47, 56, 8, 9, 62, 34, 63, 26, 52, 84, 11, 98,
                    41, 83, 16, 28, 90, 83, 40, 18, 23, 28, 75, 54, 34, 4, 20, 5,
                  

Обновление завершено!


In [31]:
def sinus_1(x):
    return np.sin(x)

def sinus_2(x):
    return np.sin(2 * x / 3)

def sinus_3(x):
    return np.sin(2 * x)

def sinus_4(x):
    return np.sin(2 / (1 + x))

In [32]:
data1, data2, data3, data4 = [0], [0], [0], [0]
dt = 0.1
t = 0

#while x := str(input("Waiting for command: ")).lower() != "q":
while True:
    x_time.append(t)
    data1.append(sinus_1(t))
    data2.append(sinus_2(t))
    data3.append(sinus_3(t))
    data4.append(sinus_4(t))
    t += dt
    
    if len(x_time) > 100:
        data1 = data1[1:]
        data2 = data2[1:]
        data3 = data3[1:]
        data4 = data4[1:]
    print(round(t, 2), "\t", data1[-1], "\t", data2[-1], "\t", data3[-1], "\t", data4[-1])
    time.sleep(dt)

NameError: name 'x_time' is not defined

In [ ]:
t = 0
x_time = []
dt = 0.1

fig = go.Figure()

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('График 1', 'График 2', 'График 3', 'График 4'),
    vertical_spacing=0.1
)

fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=2, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=3, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='lines'), row=4, col=1)

fig.update_layout(
#    title='4 графика',
    height=1200,
    showlegend=False,
)

for i in range(1, 5):
    fig.update_xaxes(range=[0, 10], row=i, col=1)
    fig.update_yaxes(range=[-2, 2], row=i, col=1)

fig = go.FigureWidget(fig)
display(fig)


while True:
    x_time.append(t)
    data1.append(sinus_1(t))
    data2.append(sinus_2(t))
    data3.append(sinus_3(t))
    data4.append(sinus_4(t))
    t += dt
    
    if len(x_time) > 100:
        data1 = data1[1:]
        data2 = data2[1:]
        data3 = data3[1:]
        data4 = data4[1:]

    with fig.batch_update():
        
        fig.data[0].x = x_time
        fig.data[0].y = data1
        
        fig.data[1].x = x_time
        fig.data[1].y = data2
        
        fig.data[2].x = x_time
        fig.data[2].y = data3
        
        fig.data[3].x = x_time
        fig.data[3].y = data4

    time.sleep(dt)
    

In [ ]:
while True:
    command = str(input("Waiting for command: "))
    if command.lower() == "q":
        break

In [ ]:
# Инициализация данных для 4 графиков
data1 = [np.random.randint(0, 100) for _ in range(100)]
data2 = [np.random.randint(0, 100) for _ in range(100)]
data3 = [np.random.randint(0, 100) for _ in range(100)]
data4 = [np.random.randint(0, 100) for _ in range(100)]

# Создаем подграфики 2x2
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('График 1', 'График 2', 'График 3', 'График 4'),
    vertical_spacing=0.1
)

# Добавляем трассы для каждого графика
fig.add_trace(go.Scatter(x=list(range(100)), y=data1, mode='lines', name='График 1'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data2, mode='lines', name='График 2'), row=2, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data3, mode='lines', name='График 3'), row=3, col=1)
fig.add_trace(go.Scatter(x=list(range(100)), y=data4, mode='lines', name='График 4'), row=4, col=1)

# Настройка внешнего вида
fig.update_layout(
    title='4 графика в реальном времени',
    height=1200,
    showlegend=False,
    template='plotly_white'
)

# Настройка осей для каждого подграфика
fig.update_xaxes(title_text="Индекс", row=1, col=1)
fig.update_xaxes(title_text="Индекс", row=2, col=1)
fig.update_xaxes(title_text="Индекс", row=3, col=1)
fig.update_xaxes(title_text="Индекс", row=4, col=1)
fig.update_yaxes(title_text="Значение", row=1, col=1)
fig.update_yaxes(title_text="Значение", row=2, col=1)
fig.update_yaxes(title_text="Значение", row=3, col=1)
fig.update_yaxes(title_text="Значение", row=4, col=1)

# Устанавливаем диапазоны для всех осей
for i in range(1, 5):
    fig.update_xaxes(range=[0, 100], row=i, col=1)
    fig.update_yaxes(range=[0, 100], row=i, col=1)

# Преобразуем в FigureWidget для интерактивного обновления
fig = go.FigureWidget(fig)

# Отображаем график
display(fig)

# Обновляем данные в цикле
for i in range(100):
    # Обновляем данные
    data1.pop(0)
    data1.append(np.random.randint(0, 100))
    
    data2.pop(0)
    data2.append(np.random.randint(0, 100))
    
    data3.pop(0)
    data3.append(np.random.randint(0, 100))
    
    data4.pop(0)
    data4.append(np.random.randint(0, 100))
    
    # Обновляем графики
    with fig.batch_update():
        fig.data[0].x = list(range(len(data1)))
        fig.data[0].y = data1
        
        fig.data[1].x = list(range(len(data2)))
        fig.data[1].y = data2
        
        fig.data[2].x = list(range(len(data3)))
        fig.data[2].y = data3
        
        fig.data[3].x = list(range(len(data4)))
        fig.data[3].y = data4
        
        # Обновляем заголовок с номером итерации
        fig.layout.title.text = f'4 графика в реальном времени (обновление {i+1}/100)'
    
    time.sleep(0.1)

print("Обновление завершено!") 

In [ ]:
from ipywidgets import Button, Output

In [ ]:
button1 = Button(description="Click!", 
                 button_style='success'
                )

out = Output()

def on_button_clicked(b):
    with out:
        out.clear_output()
        print("Click!!!")

button1.on_click(on_button_clicked)

display(button1, out)

In [ ]:
some_list = [
    [1,2,3,4],
    [2,3,4,2],
    [6,4,7,1]
]

len(some_list)

In [ ]:
some_list = some_list[1:]
some_list

In [ ]:
len(some_list[0])

In [ ]:
max(max(some_list))